# Week 4 on Colab — the baseline

No training this week. This notebook measures the model **before** you touch it: the untuned model, your system prompt, the five tools, and your own scorer.

You will run it twice: on Monday as far as **the dev cases**, to work on your prompt, and on Friday to the end, on your frozen eval set.

**Runtime → Change runtime type → T4 GPU.**

## 1. Get your code

In [ ]:
import os

REPO_URL = "..."   # the address of YOUR repo

if not os.path.exists("course") and os.path.basename(os.getcwd()) != "course":
    !git clone {REPO_URL} course
if os.path.basename(os.getcwd()) != "course":
    %cd course
!git pull -q
!pip install -q -e . pytest

## 2. The model

The course's base model for weeks 4 to 6. About 1 GB on first load.

In [ ]:
import json
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
REVISION = "7ae557604adf67be50417f59c2c2f167def9a775"

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then start again from the top."
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL, revision=REVISION, torch_dtype=torch.float16).to("cuda")
model.eval()
print(f"{sum(p.numel() for p in model.parameters()):,} parameters on", torch.cuda.get_device_name(0))

## 3. Your prompt, and what the model actually sees

The first example of the dev set, rendered. Read it as the model will: your prompt, then the five tools, then the conversation, then the start of its turn.

In [ ]:
from tunelab.dataset import check_frozen, load_jsonl, render_prompt, save_jsonl
from tunelab.generate import make_generate
from tunelab.notes import SCHEMAS
from tunelab.toolcall import run_eval, score_example

SYSTEM = Path("week-04/day-1/prompt.md").read_text().strip()
dev = load_jsonl("data/notes/dev.jsonl")
print(render_prompt(tokenizer, dev[0], SYSTEM))

generate = make_generate(model, tokenizer, SCHEMAS, SYSTEM)

## 4. The dev cases

Twenty hand-written cases, for working on your prompt. **Monday stops here.**

Read every wrong answer. Change `prompt.md` on your laptop, push, run cell 1 and this section again. Three rounds, no more: after that you are fitting the prompt to twenty examples.

In [ ]:
def show(examples, predictions, report, limit=30):
    print(f'accuracy {report["accuracy"]:.2f}   valid {report["valid_rate"]:.2f}   right tool {report["right_tool_rate"]:.2f}   '
          f'right arguments {report["right_arguments_rate"]:.2f}   refusals {report["refusal_accuracy"]}')
    for kind, row in report["by_kind"].items():
        print(f'  {kind:<10} {row["accuracy"]:.2f}  of {row["n"]}')
    wrong = [(e, p) for e, p in zip(examples, predictions) if not score_example(e, p["prediction"])["correct"]]
    print(f"\n{len(wrong)} wrong:")
    for example, row in wrong[:limit]:
        said = example["messages"][-1].get("content", "")
        print(f'\n[{example["id"]}  {example["kind"]}]  {said!r}')
        print("   wanted:", json.dumps(example["expected"], ensure_ascii=False))
        print("   got:   ", repr(row["output"]))


dev_predictions, dev_report = run_eval(generate, dev)
show(dev, dev_predictions, dev_report)

## 5. The frozen eval

**Friday only**, after `train.jsonl`, `eval.jsonl` and `eval.sha256` are committed and pushed.

Write your guess in `logs/run-log.md` first: the accuracy, and the worst kind.

In [ ]:
check_frozen("data/notes/eval.jsonl", "data/notes/eval.sha256")
held_out = load_jsonl("data/notes/eval.jsonl")

predictions, report = run_eval(generate, held_out)
show(held_out, predictions, report, limit=10)

## 6. Keep it

These two files are your baseline. Download them, put them in `data/notes/` on your laptop, and commit.

In [ ]:
save_jsonl("data/notes/baseline_predictions.jsonl", predictions)
Path("data/notes/baseline.json").write_text(json.dumps({"model": MODEL, "revision": REVISION, **report}, indent=2))

from google.colab import files

files.download("data/notes/baseline_predictions.jsonl")
files.download("data/notes/baseline.json")